In [2]:
import numpy as np
import scipy.special
from typing import List, Tuple, Dict

class HBV:
    """Simplified HBV model implementation."""

    def __init__(self):
        self.name = "HBV"

    def run_model(self, input: np.ndarray, param: List[float]) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        out, states = self._initialize_information(conceptual_inputs=input)

        # Unpack parameters
        BETA, FC, K0, K1, K2, LP, PERC, UZL, TT, CFMAX, CFR, CWH, alpha, beta = param

        # Storages
        SNOWPACK = self._initial_states["SNOWPACK"]
        MELTWATER = self._initial_states["MELTWATER"]
        SM = self._initial_states["SM"]
        SUZ = self._initial_states["SUZ"]
        SLZ = self._initial_states["SLZ"]

        # Run simulation
        for i, (p, pet, temp) in enumerate(input):
            liquid_p, snow = (p, 0) if temp > TT else (0, p)

            # Snow routine
            SNOWPACK += snow
            melt = max(min(CFMAX * (temp - TT), SNOWPACK), 0.0)
            MELTWATER += melt
            SNOWPACK -= melt

            refreezing = max(min(CFR * CFMAX * (TT - temp), MELTWATER), 0.0)
            SNOWPACK += refreezing
            MELTWATER -= refreezing

            tosoil = max(MELTWATER - (CWH * SNOWPACK), 0.0)
            MELTWATER -= tosoil

            # Soil & evapotranspiration
            soil_wetness = min(max((SM / FC) ** BETA, 0.0), 1.0)
            recharge = (liquid_p + tosoil) * soil_wetness
            SM += liquid_p + tosoil - recharge

            excess = max(SM - FC, 0.0)
            SM -= excess

            evapfactor = min(max(SM / (LP * FC), 0.0), 1.0)
            ETact = min(SM, pet * evapfactor)
            SM = max(SM - ETact, 0.0)

            # Groundwater
            SUZ += recharge + excess
            PERCact = min(SUZ, PERC)
            SUZ -= PERCact
            Q0 = K0 * max(SUZ - UZL, 0.0)
            SUZ -= Q0
            Q1 = K1 * SUZ
            SUZ -= Q1
            SLZ += PERCact
            Q2 = K2 * SLZ
            SLZ -= Q2

            # Save states
            states["SNOWPACK"][i] = SNOWPACK
            states["MELTWATER"][i] = MELTWATER
            states["SM"][i] = SM
            states["SUZ"][i] = SUZ
            states["SLZ"][i] = SLZ

            out[i] = Q0 + Q1 + Q2

        # Routing
        UH = self._gamma_routing(alpha=alpha, beta=beta, uh_len=15)
        out = self._uh_conv(discharge=out, unit_hydrograph=UH).reshape((-1, 1))
        return out, states

    def _initialize_information(self, conceptual_inputs):
        n = conceptual_inputs.shape[0]
        out = np.zeros(n)
        states = {key: np.zeros(n) for key in self._initial_states.keys()}
        return out, states

    def _gamma_routing(self, alpha: float, beta: float, uh_len: int = 10):
        x = np.arange(0.5, 0.5 + uh_len, 1)
        coeff = 1 / (beta**alpha * np.exp(scipy.special.loggamma(alpha)))
        gamma_pdf = coeff * (x ** (alpha - 1)) * np.exp(-x / beta)
        uh = gamma_pdf / np.sum(gamma_pdf)
        return uh

    def _uh_conv(self, discharge: np.ndarray, unit_hydrograph: np.ndarray):
        padding_size = unit_hydrograph.shape[0] - 1
        y = np.convolve(discharge.flatten(), unit_hydrograph, mode="full")
        return y[0:-padding_size]

    @property
    def _initial_states(self) -> Dict[str, float]:
        return {"SNOWPACK": 0.001, "MELTWATER": 0.001, "SM": 0.001, "SUZ": 0.001, "SLZ": 0.001}



In [3]:
import os
import numpy as np
import pandas as pd
import spotpy
import xarray as xr
import psutil
from joblib import Parallel, delayed
from math import sqrt
# from hbv_module import HBV  # <-- Make sure your HBV model is accessible

# ============================================================
# Efficiency Metrics
# ============================================================
def nse(sim, obs):
    return 1 - np.sum((sim - obs) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

def lognse(sim, obs):
    mask = (sim > 0) & (obs > 0)
    if np.sum(mask) < 2:
        return np.nan
    return 1 - np.sum((np.log(sim[mask]) - np.log(obs[mask])) ** 2) / \
               np.sum((np.log(obs[mask]) - np.mean(np.log(obs[mask]))) ** 2)

def kge(sim, obs):
    r = np.corrcoef(sim, obs)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    return 1 - sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


# ============================================================
# SpotPy HBV Setup Class
# ============================================================
class SpotpyHBV:
    def __init__(self, input_data, obs):
        self.input_data = input_data
        self.obs = obs.flatten()
        self.params = [
            spotpy.parameter.Uniform("BETA", 1.0, 6.0),
            spotpy.parameter.Uniform("FC", 50.0, 1000.0),
            spotpy.parameter.Uniform("K0", 0.1, 0.999),
            spotpy.parameter.Uniform("K1", 0.01, 0.5),
            spotpy.parameter.Uniform("K2", 0.001, 0.2),
            spotpy.parameter.Uniform("LP", 0.2, 1.0),
            spotpy.parameter.Uniform("PERC", 0.0, 10.0),
            spotpy.parameter.Uniform("UZL", 0.0, 100.0),
            spotpy.parameter.Uniform("TT", 0.0, 5.0),
            spotpy.parameter.Uniform("CFMAX", 0.5, 5.0),
            spotpy.parameter.Uniform("CFR", 0.0, 0.1),
            spotpy.parameter.Uniform("CWH", 0.0, 0.2),
            spotpy.parameter.Uniform("alpha", 0.0, 2.9),
            spotpy.parameter.Uniform("beta", 0.0, 6.5),
            spotpy.parameter.Uniform("precip_corr_factor", 1.0, 2.5, 1.2)
        ]

    def parameters(self):
        return spotpy.parameter.generate(self.params)

    def simulation(self, vector, input_data=None):
        hbv = HBV()  # <-- your HBV model instance
        if input_data is None:
            input_data = self.input_data

        precip_corr_factor = vector[-1]
        input_mod = input_data.copy()
        input_mod[:, 0] *= precip_corr_factor  # adjust precipitation
        discharge, _ = hbv.run_model(input_mod, vector[:-1])
        return discharge.flatten()

    def evaluation(self):
        return self.obs

    def objectivefunction(self, simulation, evaluation):
        # Multiple objectives (for SpotPy multi-like output)
        return np.array([nse(simulation, evaluation), lognse(simulation, evaluation)])


# ============================================================
# Calibration Function
# ============================================================
def calibrate_catchment_hbv(
    nc_file,
    n_samples=7000,
    output_dir="hbv_params_NSE_logNSE",
    calib_start=None,
    calib_end=None,
    val_start=None,
    val_end=None,
    n_replicates=10
):
    # -------------------------
    # Load data
    # -------------------------
    ds = xr.open_dataset(nc_file)
    df = ds.to_dataframe().reset_index()

    if "time" in df.columns:
        df = df.rename(columns={"time": "date"})
    df["date"] = pd.to_datetime(df["date"])

    catchment_name = os.path.splitext(os.path.basename(nc_file))[0]

    # Filter calibration and validation periods
    df_calib = df[(df['date'] >= calib_start) & (df['date'] <= calib_end)] if calib_start else df.copy()
    df_val = df[(df['date'] >= val_start) & (df['date'] <= val_end)] if val_start else None

    if df_calib.empty:
        print(f"⚠ Skipping {catchment_name} - no calibration data")
        return

    input_calib = df_calib[['total_precipitation_sum', 'potential_evaporation_sum', 'temperature']].values
    obs_calib = df_calib['discharge_spec'].values.flatten()

    input_val = df_val[['total_precipitation_sum', 'potential_evaporation_sum', 'temperature']].values if df_val is not None else None
    obs_val = df_val['discharge_spec'].values.flatten() if df_val is not None else None

    os.makedirs(output_dir, exist_ok=True)
    replicate_results = []

    # -------------------------
    # Run replicates
    # -------------------------
    for rep in range(n_replicates):
        print(f"{catchment_name} - Replicate {rep+1}/{n_replicates}")
        spotpy_setup = SpotpyHBV(input_calib, obs_calib)
        sampler = spotpy.algorithms.sceua(spotpy_setup, dbname=None, dbformat="ram")
        sampler.sample(n_samples)
        results = sampler.getdata()

        if results is None or len(results) == 0:
            print(f"⚠ {catchment_name} rep {rep+1}: no results returned")
            continue

        results_df = pd.DataFrame(results)

        # Detect likelihood columns robustly
        like_cols = [c for c in results_df.columns if c.startswith("like")]
        if not like_cols:
            for fallback in ["like1", "like"]:
                if fallback in results_df.columns:
                    like_cols = [fallback]
                    break

        if not like_cols:
            print(f"⚠ {catchment_name} rep {rep+1}: no 'like' columns found. Columns: {list(results_df.columns)}")
            continue

        # Combine multiple objective functions (sum)
        combined_like = results_df[like_cols].sum(axis=1)
        best_idx = combined_like.idxmax()

        # Detect parameter columns dynamically
        param_cols = [c for c in results_df.columns if c.startswith("par")]
        if not param_cols:
            try:
                param_cols = ['par' + p.name for p in spotpy_setup.params]
            except Exception:
                print(f"⚠ {catchment_name} rep {rep+1}: couldn't find parameter columns")
                continue

        best_vector = results_df.loc[best_idx, param_cols].values.tolist()

        # -------------------------
        # Calibration Simulation
        # -------------------------
        sim_calib = spotpy_setup.simulation(best_vector)
        nse_cal = nse(sim_calib, obs_calib)
        lognse_cal = lognse(sim_calib, obs_calib)
        kge_cal = kge(sim_calib, obs_calib)

        # -------------------------
        # Validation Simulation
        # -------------------------
        if df_val is not None and input_val is not None:
            sim_val = spotpy_setup.simulation(best_vector, input_data=input_val)
            nse_val = nse(sim_val, obs_val)
            lognse_val = lognse(sim_val, obs_val)
            kge_val = kge(sim_val, obs_val)
        else:
            sim_val = None
            nse_val = lognse_val = kge_val = np.nan

        # Save simulated flows
        df_cal_out = df_calib.copy()
        df_cal_out["simulated_flow"] = sim_calib
        df_cal_out.to_csv(os.path.join(output_dir, f"{catchment_name}_rep{rep+1}_cal.csv"), index=False)

        if df_val is not None and sim_val is not None:
            df_val_out = df_val.copy()
            df_val_out["simulated_flow"] = sim_val
            df_val_out.to_csv(os.path.join(output_dir, f"{catchment_name}_rep{rep+1}_val.csv"), index=False)

        # Store metrics
        par_dict = {f"par{i+1}": best_vector[i] for i in range(len(best_vector))}
        replicate_results.append({
            "replicate": rep + 1,
            "NSE_cal": nse_cal,
            "logNSE_cal": lognse_cal,
            "KGE_cal": kge_cal,
            "NSE_val": nse_val,
            "logNSE_val": lognse_val,
            "KGE_val": kge_val,
            **par_dict
        })

    # -------------------------
    # Save summary results
    # -------------------------
    if replicate_results:
        df_metrics = pd.DataFrame(replicate_results)
        df_metrics.to_csv(os.path.join(output_dir, f"{catchment_name}_replicates_metrics.csv"), index=False)
        print(f"✅ {catchment_name}: all replicates saved ({len(replicate_results)})")
    else:
        print(f"⚠ {catchment_name}: no valid replicates produced")


# ============================================================
# Safe Parallel Execution
# ============================================================
def get_safe_njobs(min_ram_per_job_gb=2.5):
    physical_cores = psutil.cpu_count(logical=False)
    available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)
    max_jobs_by_memory = int(available_ram_gb // min_ram_per_job_gb)
    safe_jobs = min(physical_cores, max_jobs_by_memory)
    return max(1, safe_jobs)


# ============================================================
# Run all catchments
# ============================================================
catchment_folder = "fill_data/gb_671"
catchment_files = [
    os.path.join(catchment_folder, f)
    for f in os.listdir(catchment_folder)
    if f.endswith(".nc")
]

n_jobs = get_safe_njobs(min_ram_per_job_gb=2.5)

Parallel(n_jobs=n_jobs, verbose=5)(
    delayed(calibrate_catchment_hbv)(
        nc_file,
        n_samples=7000,
        output_dir="hbv_params_NSE_logNSE",
        calib_start="1999-01-01",
        calib_end="2008-12-31",
        val_start="2009-01-01",
        val_end="2014-12-31",
        n_replicates=10
    )
    for nc_file in catchment_files
)

print("\n✅ All catchments processed in parallel safely.")


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed: 62.8min
[Parallel(n_jobs=16)]: Done 130 tasks      | elapsed: 183.0min
[Parallel(n_jobs=16)]: Done 256 tasks      | elapsed: 347.3min
[Parallel(n_jobs=16)]: Done 418 tasks      | elapsed: 575.3min
[Parallel(n_jobs=16)]: Done 616 tasks      | elapsed: 843.1min



✅ All catchments processed in parallel safely.


[Parallel(n_jobs=16)]: Done 671 out of 671 | elapsed: 906.2min finished
